# NIFTY Options IV Surface Reconstruction





In [1]:
!pip install lightgbm xgboost scipy scikit-learn --quiet


In [2]:
import numpy as np
import pandas as pd
import warnings
import re
import os
import glob

from scipy.interpolate import RBFInterpolator, interp1d, PchipInterpolator
from scipy.ndimage import gaussian_filter1d
from scipy.optimize import curve_fit
from sklearn.metrics import mean_squared_error
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import xgboost as xgb

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 10)
SEED = 42
np.random.seed(SEED)
print("All imports OK")


All imports OK


In [3]:
FILLED_CSV   = "filled_dataset.csv"
SOLUTION_CSV = "submission.csv"

# Auto-detect dataset.csv anywhere under /kaggle/input
matches = glob.glob('/kaggle/input/**/dataset.csv', recursive=True)
if matches:
    DATASET_CSV = matches[0]
    print(f"Found dataset: {DATASET_CSV}")
else:
    DATASET_CSV = "/kaggle/input/finclub-open-project-26/dataset.csv"
    print(f"Using default path: {DATASET_CSV}")


Found dataset: /kaggle/input/competitions/finclub-open-project-26/dataset.csv


In [4]:
df_raw = pd.read_csv(DATASET_CSV)
df_raw["datetime"] = pd.to_datetime(df_raw["datetime"], dayfirst=True)
df_raw = df_raw.sort_values("datetime").reset_index(drop=True)

OPTION_PAT  = re.compile(r"^NIFTY(\d{2}[A-Z]{3}\d{2})(\d+)(CE|PE)$")
option_cols = [c for c in df_raw.columns if OPTION_PAT.match(c)]

original_missing = df_raw[option_cols].isna().copy()

print(f"Shape          : {df_raw.shape}")
print(f"Option columns : {len(option_cols)}")
print(f"Date range     : {df_raw['datetime'].min()} -> {df_raw['datetime'].max()}")
print(f"Total missing  : {original_missing.sum().sum():,}  ({original_missing.mean().mean():.1%})")


Shape          : (975, 30)
Option columns : 28
Date range     : 2026-01-07 09:15:00 -> 2026-01-27 15:25:00
Total missing  : 5,460  (20.0%)


In [5]:
meta = {}
for col in option_cols:
    m = OPTION_PAT.match(col)
    if m:
        exp_str  = m.group(1)
        strike   = int(m.group(2))
        opt_type = m.group(3)
        exp_date = pd.to_datetime(exp_str, format="%d%b%y")
        meta[col] = dict(expiry_str=exp_str, strike=strike,
                         option_type=opt_type, expiry_date=exp_date)

meta_df = pd.DataFrame(meta).T
meta_df["strike"] = meta_df["strike"].astype(int)

print("Expiries & counts:")
print(meta_df.groupby(["expiry_str", "option_type"])["strike"]
      .agg(["count", "min", "max"]).to_string())


Expiries & counts:
                        count    min    max
expiry_str option_type                     
27JAN26    CE              14  25200  26500
           PE              14  23800  25100


In [6]:
# =============================================================================
# STEP 1 — 1D RBF Within-Timestamp Interpolation
# For each timestamp row, interpolate missing IVs across strikes.

# =============================================================================

def rbf_fill_row(iv_arr, strikes, smoothing=1e-5):
    obs = ~np.isnan(iv_arr)
    if obs.sum() < 2:
        return iv_arr
    try:
        rbf = RBFInterpolator(strikes[obs].reshape(-1, 1), iv_arr[obs],
                              kernel="thin_plate_spline", smoothing=smoothing)
        result = iv_arr.copy()
        result[~obs] = np.clip(rbf(strikes[~obs].reshape(-1, 1)), 0.0, None)
        return result
    except Exception:
        f = interp1d(strikes[obs], iv_arr[obs], kind="linear",
                     fill_value=(iv_arr[obs][0], iv_arr[obs][-1]),
                     bounds_error=False)
        result = iv_arr.copy()
        result[~obs] = np.clip(f(strikes[~obs]), 0.0, None)
        return result


def step1_rbf(df_in, meta_df):
    df = df_in.copy()
    for (exp, otype), grp in meta_df.groupby(["expiry_str", "option_type"]):
        tickers   = list(grp.index)
        strikes   = grp["strike"].values.astype(float)
        order     = np.argsort(strikes)
        tickers_s = [tickers[i] for i in order]
        strikes_s = strikes[order]
        mat = df[tickers_s].values.astype(float)
        for r in range(len(mat)):
            row = mat[r]
            if np.isnan(row).any() and not np.isnan(row).all():
                mat[r] = rbf_fill_row(row, strikes_s)
        df[tickers_s] = mat
    return df


print("Step 1: 1D RBF within-timestamp interpolation ...")
df1 = step1_rbf(df_raw.copy(), meta_df)
m1  = df1[option_cols].isna().sum().sum()
print(f"  Remaining missing: {m1:,}")


Step 1: 1D RBF within-timestamp interpolation ...
  Remaining missing: 0


In [7]:
# =============================================================================
# STEP 2 — SVI Parametric Fit + PCHIP Monotone Fill
# LEAKAGE: SAFE — uses only same-row observed data.
# =============================================================================

def svi_raw(k, a, b, rho, m, sigma):
    return a + b * (rho * (k - m) + np.sqrt((k - m)**2 + sigma**2))

def fit_svi_slice(log_m_obs, tv_obs):
    if len(log_m_obs) < 4:
        return None
    try:
        popt, _ = curve_fit(
            svi_raw, log_m_obs, tv_obs,
            p0=[np.mean(tv_obs)*0.8, 0.1, -0.3, 0.0, 0.1],
            bounds=([-1, 1e-4, -0.999, -2, 1e-4],
                    [ 1, 2.0,   0.999,  2, 2.0]),
            maxfev=5000, method="trf"
        )
        return popt
    except Exception:
        return None

def step2_svi_pchip(df_in, df_after_rbf, meta_df):
    df    = df_after_rbf.copy()
    spots = df_in["underlying_price"].values.astype(float)

    for (exp, otype), grp in meta_df.groupby(["expiry_str", "option_type"]):
        tickers   = list(grp.index)
        strikes   = grp["strike"].values.astype(float)
        order     = np.argsort(strikes)
        tickers_s = [tickers[i] for i in order]
        strikes_s = strikes[order]
        exp_date  = grp["expiry_date"].iloc[0]
        times     = df_in["datetime"].values
        mat       = df[tickers_s].values.astype(float)

        for r in range(len(df)):
            row = mat[r]
            if not np.isnan(row).any():
                continue

            spot = spots[r]
            dt   = pd.Timestamp(times[r])
            T    = max((exp_date - dt).days / 365.0, 1/365.0)
            log_m_all = np.log(spot / strikes_s)
            obs  = ~np.isnan(row)

            if obs.sum() >= 4:
                tv_obs = (row[obs] ** 2) * T
                popt   = fit_svi_slice(log_m_all[obs], tv_obs)
                if popt is not None:
                    miss = np.isnan(row)
                    tv_p = svi_raw(log_m_all[miss], *popt)
                    row[miss] = np.clip(np.sqrt(np.clip(tv_p / T, 0, None)), 0, None)

            # PCHIP for any still-missing after SVI
            still_miss = np.isnan(row)
            if still_miss.any() and (~still_miss).sum() >= 3:
                try:
                    pchip = PchipInterpolator(
                        strikes_s[~still_miss], row[~still_miss], extrapolate=True
                    )
                    row[still_miss] = np.clip(pchip(strikes_s[still_miss]), 0.0, None)
                except Exception:
                    pass

            mat[r] = np.clip(row, 0.0, None)

        df[tickers_s] = mat
    return df


print("Step 2: SVI parametric fit + PCHIP monotone fill ...")
df2 = step2_svi_pchip(df_raw, df1, meta_df)
m2  = df2[option_cols].isna().sum().sum()
print(f"  Remaining missing: {m2:,}")


Step 2: SVI parametric fit + PCHIP monotone fill ...
  Remaining missing: 0


In [8]:
# =============================================================================
# STEP 3 — Causal Forward-Fill (max_gap=12)
# =============================================================================

def step3_causal_fill(df_in, option_cols, max_gap=12):
    df = df_in.copy()
    for col in option_cols:
        df[col] = df[col].ffill(limit=max_gap).bfill(limit=2)
    return df


print("Step 3: Causal forward-fill (max_gap=12) ...")
df3 = step3_causal_fill(df2, option_cols, max_gap=12)
m3  = df3[option_cols].isna().sum().sum()
print(f"  Remaining missing: {m3:,}")


Step 3: Causal forward-fill (max_gap=12) ...
  Remaining missing: 0


In [9]:
# =============================================================================
# STEP 4 — ML Feature Engineering (with SABR parameters)
# =============================================================================

def fit_sabr_approx(strikes_arr, iv_arr, spot, T):
    """Approximate SABR: returns (alpha=ATM vol, rho=skew, nu=curvature)."""
    obs = ~np.isnan(iv_arr)
    if obs.sum() < 4 or T <= 0 or spot <= 0:
        return np.nan, np.nan, np.nan
    try:
        lm  = np.log(spot / strikes_arr[obs])
        ivs = iv_arr[obs]
        sort_idx = np.argsort(lm)
        atm_iv   = float(np.interp(0.0, lm[sort_idx], ivs[sort_idx]))
        c1       = np.polyfit(lm, ivs, 1)
        rho_p    = float(np.clip(c1[0] / (atm_iv + 1e-8), -5, 5))
        nu_p     = 0.0
        if len(lm) >= 3:
            c2   = np.polyfit(lm, ivs, 2)
            nu_p = float(abs(c2[0]) / (atm_iv + 1e-8))
        return atm_iv, rho_p, nu_p
    except Exception:
        return np.nan, np.nan, np.nan


def build_features(df_filled, df_orig, meta_df, option_cols):
    spots     = df_filled["underlying_price"].values.astype(float)
    datetimes = df_filled["datetime"].values

    ce_cols = [c for c in option_cols if meta_df.loc[c, "option_type"] == "CE"]
    pe_cols = [c for c in option_cols if meta_df.loc[c, "option_type"] == "PE"]
    ce_med  = df_filled[ce_cols].median(axis=1).values
    pe_med  = df_filled[pe_cols].median(axis=1).values
    row_std = df_filled[option_cols].std(axis=1).values

    # Pre-compute SABR params per (row, expiry, option_type)
    print("  Computing SABR parameters ...")
    sabr_cache = {}
    for (exp, otype), grp in meta_df.groupby(["expiry_str", "option_type"]):
        tickers_g = list(grp.index)
        strikes_g = grp["strike"].values.astype(float)
        order     = np.argsort(strikes_g)
        tickers_s = [tickers_g[i] for i in order]
        strikes_s = strikes_g[order]
        exp_date  = grp["expiry_date"].iloc[0]
        mat       = df_filled[tickers_s].values.astype(float)
        for r in range(len(df_filled)):
            T = max((exp_date - pd.Timestamp(datetimes[r])).days, 1) / 365.0
            a, rh, nu = fit_sabr_approx(strikes_s, mat[r], spots[r], T)
            sabr_cache[(r, exp, otype)] = (a, rh, nu)

    records = []
    for col in option_cols:
        strike   = float(meta_df.loc[col, "strike"])
        otype    = meta_df.loc[col, "option_type"]
        otype_n  = 0 if otype == "CE" else 1
        exp_date = meta_df.loc[col, "expiry_date"]
        exp_str  = meta_df.loc[col, "expiry_str"]
        iv_arr   = df_filled[col].values.astype(float)
        orig_arr = df_orig[col].values

        for i in range(len(df_filled)):
            spot = spots[i]
            dt   = pd.Timestamp(datetimes[i])
            tte  = max((exp_date - dt).days, 0)
            logm = np.log(spot / strike) if (spot > 0 and strike > 0) else 0.0

            l1  = iv_arr[i-1]  if i >= 1  else np.nan
            l2  = iv_arr[i-2]  if i >= 2  else np.nan
            l3  = iv_arr[i-3]  if i >= 3  else np.nan
            l5  = iv_arr[i-5]  if i >= 5  else np.nan
            l6  = iv_arr[i-6]  if i >= 6  else np.nan
            l10 = iv_arr[i-10] if i >= 10 else np.nan

            past  = iv_arr[max(0, i-10):i]
            pmean = np.nanmean(past) if len(past) > 0 else np.nan
            pstd  = np.nanstd(past)  if len(past) > 1 else 0.0
            pmin  = np.nanmin(past)  if len(past) > 0 else np.nan
            pmax  = np.nanmax(past)  if len(past) > 0 else np.nan
            roc   = ((iv_arr[i-1] - iv_arr[i-3]) / iv_arr[i-3]
                     if i >= 3 and iv_arr[i-3] > 0 else 0.0)

            sa, srho, snu = sabr_cache.get((i, exp_str, otype), (np.nan, np.nan, np.nan))

            records.append({
                "col"        : col,
                "ts_idx"     : i,
                "strike"     : strike,
                "logm"       : logm,
                "tte"        : tte,
                "tte_sqrt"   : np.sqrt(tte) if tte > 0 else 0.0,
                "tte_inv"    : 1.0 / (tte + 1),
                "otype"      : otype_n,
                "spot"       : spot,
                "k_s"        : strike / spot if spot > 0 else 1.0,
                "l1"         : l1,
                "l2"         : l2,
                "l3"         : l3,
                "l5"         : l5,
                "l6"         : l6,
                "l10"        : l10,
                "pmean"      : pmean,
                "pstd"       : pstd,
                "pmin"       : pmin,
                "pmax"       : pmax,
                "roc"        : roc,
                "cs_ref"     : ce_med[i] if otype_n == 0 else pe_med[i],
                "row_std"    : row_std[i],
                "sabr_alpha" : sa,
                "sabr_rho"   : srho,
                "sabr_nu"    : snu,
                "iv"         : iv_arr[i],
                "orig_obs"   : not pd.isna(orig_arr[i]),
            })

    return pd.DataFrame(records)


FEAT_COLS = [
    "strike", "logm", "tte", "tte_sqrt", "tte_inv",
    "otype", "spot", "k_s",
    "l1", "l2", "l3", "l5", "l6", "l10",
    "pmean", "pstd", "pmin", "pmax", "roc",
    "cs_ref", "row_std",
    "sabr_alpha", "sabr_rho", "sabr_nu",
]

still_missing = df3[option_cols].isna().sum().sum()
print(f"Cells requiring ML prediction: {still_missing:,}")


Cells requiring ML prediction: 0


In [10]:
# =============================================================================
# STEP 4 (continued) — LightGBM + XGBoost + MLP Ensemble
# =============================================================================
df4 = df3.copy()

if still_missing > 0:
    print("Building feature matrix ...")
    ml_df    = build_features(df3, df_raw, meta_df, option_cols)
    train_df = ml_df[ml_df["orig_obs"] & ml_df["iv"].notna()].copy()
    pred_df  = ml_df[ml_df["iv"].isna()].copy()
    print(f"Train: {len(train_df):,}  |  Predict: {len(pred_df):,}")

    ts_thresh = int(train_df["ts_idx"].quantile(0.80))
    t_mask = train_df["ts_idx"] <  ts_thresh
    v_mask = train_df["ts_idx"] >= ts_thresh

    X_tr   = train_df[FEAT_COLS][t_mask].values
    y_tr   = train_df["iv"][t_mask].values
    X_val  = train_df[FEAT_COLS][v_mask].values
    y_val  = train_df["iv"][v_mask].values
    X_pred = pred_df[FEAT_COLS].values
    print(f"Train size: {len(X_tr):,}  |  Val size: {len(X_val):,}")

    # ── LightGBM ─────────────────────────────────────────────────────────
    lgb_tr = lgb.Dataset(X_tr, label=y_tr, feature_name=FEAT_COLS)
    lgb_v  = lgb.Dataset(X_val, label=y_val, reference=lgb_tr)
    lgb_p  = {
        "objective": "regression", "metric": "mse",
        "learning_rate": 0.008, "num_leaves": 255,
        "min_child_samples": 10, "feature_fraction": 0.9,
        "bagging_fraction": 0.9, "bagging_freq": 5,
        "reg_alpha": 0.01, "reg_lambda": 0.05,
        "seed": SEED, "verbose": -1,
    }
    print("\nTraining LightGBM ...")
    lgb_model = lgb.train(lgb_p, lgb_tr, num_boost_round=5000,
                          valid_sets=[lgb_v],
                          callbacks=[lgb.early_stopping(200, verbose=False),
                                     lgb.log_evaluation(500)])
    lgb_val_pred = lgb_model.predict(X_val)
    lgb_mse = mean_squared_error(y_val, lgb_val_pred)
    print(f"LightGBM val MSE: {lgb_mse:.8f}")

    # ── XGBoost ──────────────────────────────────────────────────────────
    dtrain = xgb.DMatrix(X_tr,   label=y_tr,  feature_names=FEAT_COLS)
    dval   = xgb.DMatrix(X_val,  label=y_val, feature_names=FEAT_COLS)
    dpred  = xgb.DMatrix(X_pred,              feature_names=FEAT_COLS)
    xgb_p  = {
        "objective": "reg:squarederror", "eval_metric": "rmse",
        "max_depth": 8, "eta": 0.008,
        "subsample": 0.9, "colsample_bytree": 0.9,
        "alpha": 0.01, "lambda": 0.05,
        "seed": SEED, "tree_method": "hist", "verbosity": 0,
    }
    print("\nTraining XGBoost ...")
    xgb_model = xgb.train(xgb_p, dtrain, num_boost_round=5000,
                           evals=[(dval, "val")],
                           early_stopping_rounds=200, verbose_eval=500)
    xgb_val_pred = xgb_model.predict(dval)
    xgb_mse = mean_squared_error(y_val, xgb_val_pred)
    print(f"XGBoost val MSE:  {xgb_mse:.8f}")

    # ── MLP ──────────────────────────────────────────────────────────────
    print("\nTraining MLP ...")
    scaler   = StandardScaler()
    X_tr_s   = scaler.fit_transform(np.nan_to_num(X_tr,   nan=0.0))
    X_val_s  = scaler.transform(    np.nan_to_num(X_val,  nan=0.0))
    X_pred_s = scaler.transform(    np.nan_to_num(X_pred, nan=0.0))
    mlp = MLPRegressor(
        hidden_layer_sizes=(512, 256, 128, 64),
        activation="relu", solver="adam",
        learning_rate_init=5e-4, max_iter=500,
        early_stopping=True, validation_fraction=0.1,
        n_iter_no_change=30, random_state=SEED, verbose=False,
    )
    mlp.fit(X_tr_s, y_tr)
    mlp_val_pred = mlp.predict(X_val_s)
    mlp_mse = mean_squared_error(y_val, mlp_val_pred)
    print(f"MLP val MSE:      {mlp_mse:.8f}")

    # ── Inverse-MSE weighted ensemble ────────────────────────────────────
    inv     = {"lgb": 1/lgb_mse, "xgb": 1/xgb_mse, "mlp": 1/mlp_mse}
    tot     = sum(inv.values())
    w_lgb, w_xgb, w_mlp = inv["lgb"]/tot, inv["xgb"]/tot, inv["mlp"]/tot
    print(f"\nWeights -> LGB:{w_lgb:.3f}  XGB:{w_xgb:.3f}  MLP:{w_mlp:.3f}")

    p_lgb = lgb_model.predict(X_pred)
    p_xgb = xgb_model.predict(dpred)
    p_mlp = mlp.predict(X_pred_s)
    ens   = np.clip(w_lgb*p_lgb + w_xgb*p_xgb + w_mlp*p_mlp, 0.0, None)

    for (col, ts_i), val in zip(zip(pred_df["col"], pred_df["ts_idx"]), ens):
        df4.at[ts_i, col] = val

    # ── Second pass: median ensemble for uncertain cells (no lag context) ─
    uncertain_mask = pred_df["l1"].isna()
    n_unc = uncertain_mask.sum()
    print(f"\nSecond pass for {n_unc:,} uncertain cells (no lag1 context) ...")
    if n_unc > 0:
        X_unc   = pred_df[FEAT_COLS][uncertain_mask].values
        X_unc_s = scaler.transform(np.nan_to_num(X_unc, nan=0.0))
        pu_lgb  = lgb_model.predict(X_unc)
        pu_xgb  = xgb_model.predict(xgb.DMatrix(X_unc, feature_names=FEAT_COLS))
        pu_mlp  = mlp.predict(X_unc_s)
        pu_med  = np.clip(
            np.median(np.column_stack([pu_lgb, pu_xgb, pu_mlp]), axis=1),
            0.0, None
        )
        unc_idx = pred_df.index[uncertain_mask]
        for k, (col, ts_i) in enumerate(
            zip(pred_df.loc[unc_idx, "col"], pred_df.loc[unc_idx, "ts_idx"])
        ):
            df4.at[ts_i, col] = pu_med[k]
        print(f"  Done.")

    print(f"\nRemaining missing after ML: {df4[option_cols].isna().sum().sum():,}")
else:
    print("No missing values left — ML step skipped.")
    scaler = None


No missing values left — ML step skipped.


In [11]:
# =============================================================================
# STEP 5 — Safety Net + CV-Tuned Gaussian Smoothing Blend
# =============================================================================
df5 = df4.copy()

# Safety net
for col in option_cols:
    if df5[col].isna().any():
        med = df5[col].median()
        df5[col] = df5[col].fillna(
            med if not pd.isna(med) else df5[option_cols].stack().median()
        )
assert df5[option_cols].isna().sum().sum() == 0, "Unfilled values remain!"

def tune_alpha(df_pred, df_orig, option_cols, original_missing,
               alphas=np.linspace(0.0, 0.5, 11), sigma=0.7):
    obs_r, obs_c = np.where(~original_missing.values)
    np.random.seed(0)
    n   = min(20000, len(obs_r))
    idx = np.random.choice(len(obs_r), n, replace=False)
    y_true = np.array([df_orig.iloc[obs_r[i]][option_cols[obs_c[i]]] for i in idx])
    best_alpha, best_mse = 0.0, np.inf
    for alpha in alphas:
        df_tmp = df_pred.copy()
        for col in option_cols:
            miss = original_missing[col].values
            if not miss.any():
                continue
            arr      = df_pred[col].values.astype(float)
            smoothed = gaussian_filter1d(arr, sigma=sigma)
            arr2     = arr.copy()
            arr2[miss] = np.clip((1-alpha)*arr[miss] + alpha*smoothed[miss], 0.0, None)
            df_tmp[col] = arr2
        y_pred = np.array([df_tmp.iloc[obs_r[i]][option_cols[obs_c[i]]] for i in idx])
        mse = mean_squared_error(y_true, y_pred)
        if mse < best_mse:
            best_mse, best_alpha = mse, alpha
    print(f"  Best ALPHA={best_alpha:.2f}  (val MSE={best_mse:.8f})")
    return best_alpha

print("Tuning smoothing ALPHA by CV on observed cells ...")
ALPHA = tune_alpha(df5, df_raw, option_cols, original_missing)

SIGMA = 0.7
for col in option_cols:
    miss_mask = original_missing[col].values
    if not miss_mask.any():
        continue
    arr      = df5[col].values.astype(float)
    smoothed = gaussian_filter1d(arr, sigma=SIGMA)
    arr[miss_mask] = np.clip(
        (1-ALPHA)*arr[miss_mask] + ALPHA*smoothed[miss_mask], 0.0, None
    )
    df5[col] = arr

print(f"Step 5: Smoothing applied (ALPHA={ALPHA:.2f})")
print(f"Final missing: {df5[option_cols].isna().sum().sum()}")


Tuning smoothing ALPHA by CV on observed cells ...
  Best ALPHA=0.00  (val MSE=0.00000000)
Step 5: Smoothing applied (ALPHA=0.00)
Final missing: 0


In [12]:
np.random.seed(0)
obs_r, obs_c = np.where(~original_missing.values)
n_sample = min(100_000, len(obs_r))
idx = np.random.choice(len(obs_r), n_sample, replace=False)
y_true_pv = [df_raw.iloc[obs_r[i]][option_cols[obs_c[i]]] for i in idx]
y_pred_pv = [df5.iloc[obs_r[i]][option_cols[obs_c[i]]]    for i in idx]
pv_mse = mean_squared_error(y_true_pv, y_pred_pv)
print(f"Pseudo-val MSE (observed cells): {pv_mse:.8f}")
print(f"Pseudo-val RMSE                : {np.sqrt(pv_mse):.6f}")
print()
print("Reference LB scores (public 30%):")
print("  #1  : 0.0000332638")
print("  #10 : 0.0000372757")
print("  You : 0.0000557566  (before this update)")


Pseudo-val MSE (observed cells): 0.00000000
Pseudo-val RMSE                : 0.000000

Reference LB scores (public 30%):
  #1  : 0.0000332638
  #10 : 0.0000372757
  You : 0.0000557566  (before this update)


In [13]:
# =============================================================================
# STEP 6 — Convexity Enforcement (No-Arbitrage, 4 passes)
# =============================================================================

def enforce_convexity(df_in, meta_df, original_missing, n_passes=4):
    df = df_in.copy()
    for _ in range(n_passes):
        for (exp, otype), grp in meta_df.groupby(["expiry_str", "option_type"]):
            tickers   = list(grp.index)
            strikes   = grp["strike"].values.astype(float)
            order     = np.argsort(strikes)
            tickers_s = [tickers[i] for i in order]
            for r in range(len(df)):
                iv   = df.loc[r, tickers_s].values.astype(float)
                miss = original_missing.loc[r, tickers_s].values
                for j in range(1, len(iv) - 1):
                    if not miss[j]:
                        continue
                    avg = 0.5 * (iv[j-1] + iv[j+1])
                    if not np.isnan(avg):
                        if iv[j] > avg * 1.15:
                            iv[j] = 0.85 * iv[j] + 0.15 * avg
                        elif iv[j] < avg * 0.85:
                            iv[j] = 0.85 * iv[j] + 0.15 * avg
                df.loc[r, tickers_s] = np.clip(iv, 0.0, None)
    return df


print("Step 6: Convexity enforcement (4 passes) ...")
df6 = enforce_convexity(df5, meta_df, original_missing, n_passes=4)
print(f"  Done. All IVs non-negative: {(df6[option_cols] >= 0).all().all()}")


Step 6: Convexity enforcement (4 passes) ...
  Done. All IVs non-negative: True


In [14]:
df_save = df6.copy()
df_save["datetime"] = df_save["datetime"].dt.strftime("%d-%m-%Y %H:%M")
df_save.to_csv(FILLED_CSV, index=False)
print(f"Saved: {FILLED_CSV}")


Saved: filled_dataset.csv


In [15]:
# Competition submission generator — verbatim from contest page
ORIGINAL_DATASET_PATH = DATASET_CSV
SEPARATOR = "||"

def generate_solution(filled_path, output_path="submission.csv"):
    original = pd.read_csv(ORIGINAL_DATASET_PATH)
    filled   = pd.read_csv(filled_path)
    feature_cols = [c for c in original.columns if c != "datetime"]
    rows = []
    for col in feature_cols:
        was_missing = original[col].isna()
        for idx in original.index[was_missing]:
            dt  = original.loc[idx, "datetime"]
            uid = f"{dt}{SEPARATOR}{col}"
            val = filled.loc[idx, col]
            rows.append({"id": uid, "value": val})
    solution = pd.DataFrame(rows, columns=["id", "value"])
    solution = solution.sort_values("id").reset_index(drop=True)
    solution.to_csv(output_path, index=False)
    print(f"Solution saved -> {output_path}  ({len(solution)} rows)")

generate_solution(FILLED_CSV, SOLUTION_CSV)


Solution saved -> submission.csv  (5460 rows)


In [16]:
sol = pd.read_csv(SOLUTION_CSV)
orig_check = pd.read_csv(DATASET_CSV)
feat_cols_orig = [c for c in orig_check.columns if c != "datetime"]
expected = sum(orig_check[c].isna().sum() for c in feat_cols_orig)

print(f"[{'OK' if sol['value'].isna().sum()==0 else 'FAIL'}] NaN values    : {sol['value'].isna().sum()}")
print(f"[{'OK' if (sol['value']<0).sum()==0 else 'WARN'}] Negative IVs : {(sol['value']<0).sum()}")
print(f"[{'OK' if len(sol)==expected else 'FAIL'}] Row count     : {len(sol)} (expected {expected})")
print(f"[INFO] Value range  : [{sol['value'].min():.4f}, {sol['value'].max():.4f}]")
print(f"[INFO] Value mean   : {sol['value'].mean():.6f}")
print()
if sol['value'].isna().sum()==0 and len(sol)==expected:
    print("ALL CHECKS PASSED - safe to submit!")
else:
    print("CHECKS FAILED - review above")
print()
print(sol.head(10).to_string(index=False))


[OK] NaN values    : 0
[OK] Negative IVs : 0
[OK] Row count     : 5460 (expected 5460)
[INFO] Value range  : [0.0336, 5.7877]
[INFO] Value mean   : 0.186833

ALL CHECKS PASSED - safe to submit!

                                   id    value
07-01-2026 09:15||NIFTY27JAN2624100PE 0.163776
07-01-2026 09:15||NIFTY27JAN2625500CE 0.113462
07-01-2026 09:15||NIFTY27JAN2625800CE 0.101074
07-01-2026 09:20||NIFTY27JAN2624000PE 0.170120
07-01-2026 09:20||NIFTY27JAN2624200PE 0.160060
07-01-2026 09:20||NIFTY27JAN2624800PE 0.128104
07-01-2026 09:20||NIFTY27JAN2625000PE 0.118667
07-01-2026 09:20||NIFTY27JAN2625300CE 0.099048
07-01-2026 09:20||NIFTY27JAN2625400CE 0.111265
07-01-2026 09:20||NIFTY27JAN2625800CE 0.106830
